# Bilingual StereoSet split for Ethos

Each intervention targets one StereoSet domain in one language. The two languages share source IDs and split assignments. The registered 949 evaluation IDs stay fixed. This notebook rebuilds the split from the committed, scored item pairs and verifies the exact allocation used for training. It uses no model predictions from the study.

In [1]:
import json
import os
import sys
from pathlib import Path

from IPython.display import Markdown, display

candidates = [
    Path(os.environ.get("ETHOS_ARTIFACT_ROOT", ".")).resolve(),
    *Path.cwd().resolve().parents,
]
for directory in candidates:
    choices = [directory / "split", directory / "redistibution-artifacts/split", directory]
    split_dir = next(
        (x for x in choices if (x / "screen.py").is_file() and (x / "partition.py").is_file()), None
    )
    if split_dir:
        break
else:
    raise FileNotFoundError("Set ETHOS_ARTIFACT_ROOT to the redistibution-artifacts directory")
sys.path.insert(0, str(split_dir))
from partition import DOMAINS, PARTITIONS, build, csv_rows, pair_rows, reasons  # noqa: E402
from screen import ENCODER, RERANKER, source_digest  # noqa: E402

frozen = split_dir / "release"
audit = json.loads((frozen / "audit.json").read_text())
for filename, key in (
    ("items.csv", "items_sha256"),
    ("scores.csv.gz", "scores_sha256"),
    ("claim_links.csv.gz", "spans_sha256"),
    ("assignments.csv", "assignments_sha256"),
):
    assert source_digest(frozen / filename) == audit[key], filename
assert source_digest(split_dir / "frontier_review.csv") == audit["frontier_sha256"]
items = csv_rows(frozen / "items.csv")
assert len({r["item_id"] for r in items}) == len(items) == 2101
print(
    f"{len(items):,} matched English–Ukrainian IDs; {sum(r['protected'] == '1' for r in items)} protected evaluation IDs."
)

2,101 matched English–Ukrainian IDs; 949 protected evaluation IDs.


## Which items travel together

The screening script normalizes complete sentences using Unicode NFKC, case folding and word boundaries. It links an exact sentence in any of the three roles across items in either language. It also links identical stereotype and counter-stereotype sentence pairs after replacing their lexical targets, and imports documented links from the earlier gender pilot.

A pinned multilingual paraphrase encoder compares every item pair in English and Ukrainian. The screen uses the larger language-specific cosine similarity for the stereotype and for the mean of the stereotype and counter-stereotype similarities. An English sentence-pair model also scores every same-target pair, candidates at or above 0.600 on either bilingual signal, and the nearest neighbors. A link requires an exact or prior link, bilingual stereotype similarity of at least 0.850, bilingual paired similarity of at least 0.800, or English sentence-pair similarity of at least 0.800. Scores are rounded to thousandths before applying thresholds. These thresholds govern the sentence-level screen. They have no interpretation as calibrated probabilities of equivalence.

A second pass isolates the phrase that changes between the stereotype and counter-stereotype by removing their shared token prefix and suffix. Within a lexical target, a stereotype-phrase similarity of at least 0.820 or a two-way swapped contrast of at least 0.850 joins the pair. Across targets, both phrases must reach 0.950 for this rule. Either language can trigger a link.

A text-based audit examined the 25 closest remaining cross-split pairs per domain in the initial sentence-only partition, ranked by distance to the closest sentence-screen cutoff. The [recorded decisions](frontier_review.csv) add confirmed or plausibly linked claims. For example, the two English sentences about software developers being nerdy share the same claim despite different time references and counter-stereotypes. The audit used source text before training. No human annotation or model outcome was used.

Connected components include transitive links across targets and domains. If A links to B and B to C, all three share a component, even when A and C have different wording.

In [2]:
print("Multilingual encoder", *ENCODER)
print("English pair scorer", *RERANKER)
decisions, rebuilt = build(
    items,
    frozen / "scores.csv.gz",
    frozen / "claim_links.csv.gz",
    split_dir / "frontier_review.csv",
)
assert decisions == csv_rows(frozen / "assignments.csv")
assert all(audit[key] == value for key, value in rebuilt.items())
print(
    f"{rebuilt['all_unordered_pairs']:,} possible pairs; {rebuilt['reranked_pairs']:,} reranked; {rebuilt['linked_pairs']:,} links."
)
print(
    f"{rebuilt['total_components']:,} connected components, including {rebuilt['protected_components']:,} containing evaluation items."
)

Multilingual encoder sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 e8f8c211226b894fcb81acc59f3b34ba3efd5f42
English pair scorer cross-encoder/stsb-roberta-base d576534b67143e2c70ee9966d7fdbf5835728d13
2,206,050 possible pairs; 37,341 reranked; 5,167 links.
734 connected components, including 361 containing evaluation items.


## Allocation and coverage

Only registered evaluation IDs contribute to the benchmark score. Additional members of their components are quarantined from training and development. Among the remaining components the target is about 10% development within each domain. A stable SHA-256 order with seed 42 selects whole components when they improve the four domain counts. The balance can differ from exactly 10% because a component may span several items or domains. The same split applies to every model, training language and training seed.

In [3]:
def table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
    display(Markdown("\n".join(lines)))


counts = rebuilt["domain_counts"]
table(
    ["Domain", "Train", "Development", "Held-out", "Quarantined"],
    [[d, *[counts[d][s] for s in PARTITIONS]] for d in DOMAINS]
    + [["Total", *[sum(counts[d][s] for d in DOMAINS) for s in PARTITIONS]]],
)
assert sum(sum(c.values()) for c in counts.values()) == 2101

| Domain | Train | Development | Held-out | Quarantined |
| --- | --- | --- | --- | --- |
| gender | 34 | 4 | 110 | 105 |
| profession | 139 | 15 | 354 | 300 |
| race | 177 | 20 | 453 | 312 |
| religion | 17 | 2 | 32 | 27 |
| Total | 367 | 41 | 949 | 744 |

## Leakage checks

The checks below verify item disjointness, family disjointness, preservation of the published evaluation panel and containment of every detected link. The highest remaining cross-split scores quantify how close eligible items come to the thresholds. Splitting at the family level prevents measured intervention effects on paraphrases from being counted as transfer to unseen claims.

In [4]:
by_id = {r["item_id"]: r for r in decisions}
active = ("train", "dev", "eval")
groups = {s: {r["family"] for r in decisions if r["split"] == s} for s in active}
for i, s in enumerate(active):
    for t in active[i + 1 :]:
        assert groups[s].isdisjoint(groups[t])
protected = {r["item_id"] for r in items if r["protected"] == "1"}
assert {i for i, r in by_id.items() if r["split"] == "eval"} == protected
assert all(r["family"] in groups["eval"] for r in decisions if r["split"] == "exclude")
for pair in pair_rows(frozen / "scores.csv.gz"):
    if reasons(pair):
        assert by_id[pair["left"]]["family"] == by_id[pair["right"]]["family"]
for pair in pair_rows(frozen / "claim_links.csv.gz"):
    assert by_id[pair["left"]]["family"] == by_id[pair["right"]]["family"]
for pair in csv_rows(split_dir / "frontier_review.csv"):
    if pair["decision"] == "link":
        assert by_id[pair["left"]]["family"] == by_id[pair["right"]]["family"]
print("Detected links crossing train, development or evaluation:", rebuilt["cross_partition_links"])
print(
    "Largest retained cross-split scores:",
    {k: v / 1000 for k, v in rebuilt["largest_retained_cross_split_scores"].items()},
)
pilot = json.loads((split_dir / "pilot_links.json").read_text())
table(
    ["Earlier pilot role", *active, "exclude"],
    [
        [k, *[sum(by_id[i]["split"] == s for i in pilot[k]) for s in (*active, "exclude")]]
        for k in ("prior_train_ids", "prior_dev_ids")
    ],
)

Detected links crossing train, development or evaluation: 0
Largest retained cross-split scores: {'bilingual_stereo': 0.829, 'bilingual_pair': 0.794, 'english_sts': 0.761}


| Earlier pilot role | train | dev | eval | exclude |
| --- | --- | --- | --- | --- |
| prior_train_ids | 3 | 0 | 0 | 1 |
| prior_dev_ids | 2 | 0 | 0 | 0 |

## What reached the training executable

The independent export check compared all sixteen training and development CSV files with the frozen assignments and both source roles, in order. It reconstructed the complete held-out triplets in both languages from the registered sources. The recorded checksums bind this inspection to the actual suite. Run `python split/check_export.py --suite "$WORK/suite" "${sources[@]}"` with the four source paths from the run guide to repeat it.

In [5]:
binding = json.loads((frozen / "export_check.json").read_text())
config = json.loads((split_dir.parent / "config/stereoset_domains.json").read_text())
assert binding["registered_sources"] == config["stereoset_sources"]
for lang in ("en", "uk"):
    for domain in DOMAINS:
        for split, filename in (("train", "train/test.csv"), ("dev", "dev.csv")):
            assert binding["stream_rows"][f"{lang}/{domain}/{filename}"] == counts[domain][split]
assert binding["protected_rows_per_language"] == 949
assert len(pilot["prior_train_ids"]) == 4 and len(pilot["prior_dev_ids"]) == 2
print("Verified training export", binding["suite_sha256"])
print("Source-checked streams:", len(binding["stream_rows"]))

Verified training export 1771e0cfa56dd7dcbca547c765f9d5622a7bab2eb02db4041a262dc543be9026
Source-checked streams: 16


## Interpretation and reproduction

Automated checks can miss a paraphrase and can group distinct claims conservatively. The zero count applies to the links detectable by this screen. Benchmark publication, human validation, latent conceptual overlap and model pretraining exposure are separate questions. Every Ethos intervention starts from its pinned base checkpoint.

The committed ID inventory, pair evidence, thresholds, assignments and source hashes reproduce the released split without downloading any weights. To regenerate the semantic scores from the four registered CSV files, follow the commands in the [run guide](../README.md). Review the scored neighbors and the quarantined fraction per domain when interpreting redistribution.